---
title: Week 4 Challeng 3 KEY
subtitle: DS 2023 | Communicating with Data
---

## Overview 


Today's challenge will focus on what is perhaps the greatest data visualization in astronomy, and definitely one of the coolest scatterplots out there. 

It's called **the Hertzsprung–Russell Diagram**,.

Around 1910, Ejnar Hertzsprung and Henry Norris Russell independently plotted a collection of stars' **brightness** against their **color (temperature)**. 

Instead of a random cloud, the stars fell into distinct clusters. 

The cool thing is that these clusters tell a story about the stars the populate our universe.

Here is one version of the diagram, fully annotated:

![](https://virginia.box.com/shared/static/6s3lbg2960cu11ljr3femmglgzqkp52n.png)

[Source](http://www.atlasoftheuniverse.com/hr.html)

Watch this incredible video on how the plot is constructed from the original physical phenomena and the data abstracted from them.

In [1]:
from IPython.display import Video
Video("https://assets.science.nasa.gov/content/dam/science/missions/hubble/releases/2010/10/STScI-01EVSKXA7JN0AR357P4TZ25FA5.mp4", width=750)

[Source](https://science.nasa.gov/asset/hubble/constructing-the-hertzsprung-russell-diagram-for-globular-star-cluster/)

Today you'll rebuild it (partially) using real measurements from the European Space Agency's **Gaia** mission, which has measured positions, distances, and colors for over a billion stars.

## About the data

Our data come from [Gaia Data Release 3 (DR3)](https://gea.esac.esa.int/archive/), published by the European Space Agency in June 2022. 

It contains $28,194$ obervations (i.e. specific stars) selected with a query against the Gaia Archive. 

Here's a data dictionary for the columns in the dataset.

| Column | Units | Description |
|---|---|---|
| `source_id` | — | Unique identifier for the star in the Gaia catalog. Not a random integer: it encodes the star's HEALPix sky position, so it's stable across queries but shouldn't be used as a sort key or treated as meaningful on its own. |
| `ra` | degrees (0–360) | Right ascension in the ICRS frame — celestial longitude, at epoch J2016.0. |
| `dec` | degrees (−90 to +90) | Declination in the ICRS frame — celestial latitude, at epoch J2016.0. |
| `parallax` | milliarcseconds (mas) | The star's measured trigonometric parallax. Distance in parsecs ≈ 1000 / parallax. Values in your sample (~20–53 mas) correspond to roughly 19–50 pc, so this is a nearby-star sample. |
| `parallax_over_error` | dimensionless | Parallax divided by its formal uncertainty — a signal-to-noise ratio for the distance. Values above ~10 are usually considered reliable; yours are 500–2000, so these are exceptionally well-measured stars. |
| `phot_g_mean_mag` | magnitudes (Vega system) | Mean apparent brightness in Gaia's broad *G* band (~330–1050 nm). Lower numbers are brighter. |
| `phot_bp_mean_mag` | magnitudes | Mean apparent brightness in the blue photometer band *G*<sub>BP</sub> (~330–680 nm). |
| `phot_rp_mean_mag` | magnitudes | Mean apparent brightness in the red photometer band *G*<sub>RP</sub> (~630–1050 nm). |
| `bp_rp` | magnitudes | Color index, defined as `phot_bp_mean_mag − phot_rp_mean_mag`. Small/negative values are blue and hot; large values are red and cool. |

## Tasks

### Task 0 Get the data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
stars_url = "https://virginia.box.com/shared/static/vtwmlxmq946w54jjw1jmarkm3e8qheoi.csv"
stars = pd.read_csv(stars_url)
stars = stars.set_index('source_id')
stars

,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp
source_id,,,,,,,,
208795888023928448,80.010925,45.780002,20.970814,1146.57750,13.367318,14.668081,12.238841,2.429240
210136398855106304,88.964383,47.823791,28.063795,1312.72510,14.074352,15.661598,12.840337,2.821261
213983521321794688,84.836064,49.032156,25.110410,522.59340,16.163118,18.337704,14.808206,3.529498
231618931920424704,60.274290,43.460777,29.303638,1171.58740,8.066113,8.461499,7.499279,0.962220
232099452860704768,62.718525,43.026259,28.583804,1295.25460,11.379289,12.441178,10.355892,2.085286
...,...,...,...,...,...,...,...,...
6686843727130165888,299.240955,-42.277614,33.665575,540.05960,15.945090,18.494225,14.541685,3.952539
6686866164036956160,299.911005,-42.401192,20.862625,1112.04690,13.258162,14.639338,12.099594,2.539744
6708468750032965632,276.365829,-46.768088,26.759940,1008.97723,14.818999,16.643724,13.535386,3.108338


### Task 1

In order to produce an effective visualization, we need to create some new columns that allow to compute a star's true brightness.

This is a because a nearby dim star and a distant brilliant star can have the same apparent magnitude (brightness).

To do this, we add these columns to `stars`:

- `distance_pc`: the distance in parsecs = `1000 / stars['parallax']`
- `abs_g_mag`: the absolute G magnitude = `stars["phot_g_mean_mag"] + 5 * np.log10(stars["parallax"]) - 10`

In [ ]:
stars['distance_pc'] = 1000 / stars['parallax']
stars['abs_g_mag'] = stars['phot_g_mean_mag'] + 5 * np.log10(stars['parallax']) - 10
stars[['parallax', 'distance_pc', 'phot_g_mean_mag', 'abs_g_mag']].head()

### Task 2

Which is the closest star in the data set? 

To find out, use either `.nsmallest()` or `.sort_values()`.

In [ ]:
stars.nsmallest(1, 'distance_pc')

The closest star to the Sun is `5853498713190525696`, [Proxima Centauri](https://in-the-sky.org/data/object.php?id=TYC9010-4949-1), is about 1.3 pc away.

### Task 3 

Make a scatter plot of `stars` with the following requirements:

- Set the x axis to `bp_rp` (color).
- Set the y axis to `abs_g_mag` (absolute magnitude).
- Use small, semi-transparent points (try `s=1`, `alpha=0.3`), since there are thousands of stars.
- Set the figure size to $7$ by $8$.
- Flip the y axis, so that bright stars (small magnitudes) are at the top. This is the astronomers' convention. See hint below.

```python
ax.invert_yaxis()
```

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(stars['bp_rp'], stars['abs_g_mag'], s=1, alpha=0.3)
ax.invert_yaxis()
ax.set_xlabel('Color index $G_{BP} - G_{RP}$ (mag)')
ax.set_ylabel('Absolute magnitude $M_G$ (mag)')
ax.set_title('Hertzsprung–Russell Diagram (Gaia DR3)')
plt.show()

### Task 4

Now make the plot look like a sky full of stars:

- Use a dark background, i.e.  set `plt.style.context("dark_background")`. Hint: put this expression in a `with` statement and put the plot code in its body. That is, do this:
```python
with plt.style.context("dark_background"):
    # put plotting code here
```
- Color each point by its `bp_rp` value with a colormap that runs blue to red (try `cmap="RdYlBu_r"`).
- Bump up the figure size to $10$ by $15$.

In [ ]:
with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(10, 15))
    sc = ax.scatter(stars['bp_rp'], stars['abs_g_mag'],
                    c=stars['bp_rp'], cmap="RdYlBu_r", s=1, alpha=0.3)
    ax.invert_yaxis()
    ax.set_xlabel('Color index $G_{BP} - G_{RP}$ (mag)')
    ax.set_ylabel('Absolute magnitude $M_G$ (mag)')
    ax.set_title('Hertzsprung–Russell Diagram (Gaia DR3)')
    fig.colorbar(sc, ax=ax, label='$G_{BP} - G_{RP}$')
    plt.show()

### Task 5

Create a hexbin plot of the same two varaibles. 

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
hb = ax.hexbin(stars['bp_rp'], stars['abs_g_mag'], gridsize=80, bins='log', cmap='inferno', mincnt=1)
ax.invert_yaxis()
ax.set_xlabel('Color index $G_{BP} - G_{RP}$ (mag)')
ax.set_ylabel('Absolute magnitude $M_G$ (mag)')
ax.set_title('Hertzsprung–Russell Diagram (Gaia DR3), hexbin')
fig.colorbar(hb, ax=ax, label='Number of stars (log scale)')
plt.show()